# Eval: comprobar lo observado · 2.1.4

Input value=3 → Agent suma uno → Eval exige 4. Un control negativo exige 999 y debe fallar. Eval es explícito y transversal a Agent, System y Environment.

In [2]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'


In [3]:
class State(pydantic.BaseModel):
    value: int

@toolkit.tool
def increment(value: int) -> dict:
    return {'value': value + 1}
graph = toolkit.Graph(State)
graph.add_node('increment', increment)
graph.add_edge(toolkit.START, 'increment')
graph.add_edge('increment', toolkit.END)
agent = toolkit.Agent('counter', graph=graph)


In [4]:
evaluation = toolkit.Eval('correctness', agent, cases=({'input': {'value': 3}, 'expected': {'data': {'value': 4}}},))
passed = evaluation.run()
negative = toolkit.Eval('negative', agent, cases=({'input': {'value': 3}, 'expected': {'data': {'value': 999}}},)).run()
assert passed.ok and (not negative.ok)
print({'positive': passed.ok, 'negative_detected': not negative.ok})


{'positive': True, 'negative_detected': True}


Eval termina ante el primer caso fallido. Para medir todas las decisiones negativas, ejecutar casos independientes. Un juez fixture comprueba el protocolo; no acredita juicio semántico.